1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, f1_score
import time

from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

from retrieval.vector_search import VectorSearch
from retrieval.bm25_search import BM25Search
from retrieval.hybrid_search import HybridRetriever
from evaluation.metrics import RAGMetrics

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

Create Test Dataset

In [ ]:
# Create a test document collection
documents_data = [
    {
        "content": "Python is a high-level programming language known for its simplicity and readability.",
        "id": "doc1",
        "topic": "programming"
    },
    {
        "content": "Machine learning is a subset of artificial intelligence that learns from data.",
        "id": "doc2",
        "topic": "ai"
    },
    {
        "content": "JavaScript is widely used for web development and creating interactive websites.",
        "id": "doc3",
        "topic": "programming"
    },
    {
        "content": "Deep learning uses neural networks with multiple layers to process complex data.",
        "id": "doc4",
        "topic": "ai"
    },
    {
        "content": "SQL is a language for managing and querying relational databases.",
        "id": "doc5",
        "topic": "database"
    },
    {
        "content": "Natural language processing enables computers to understand human language.",
        "id": "doc6",
        "topic": "ai"
    },
    {
        "content": "React is a JavaScript library for building user interfaces.",
        "id": "doc7",
        "topic": "programming"
    },
    {
        "content": "MongoDB is a NoSQL database that stores data in JSON-like documents.",
        "id": "doc8",
        "topic": "database"
    },
    {
        "content": "Computer vision enables machines to interpret and understand visual information.",
        "id": "doc9",
        "topic": "ai"
    },
    {
        "content": "Docker is a platform for developing and running containerized applications.",
        "id": "doc10",
        "topic": "devops"
    }
]

documents = [
    Document(
        page_content=doc["content"],
        metadata={"id": doc["id"], "topic": doc["topic"]}
    )
    for doc in documents_data
]

print(f"Created {len(documents)} test documents")

print("\nDocument topics:")
topic_counts = pd.Series(
    [doc.metadata["topic"] for doc in documents]
).value_counts()

print(topic_counts)

Initialize Retrievers

In [ ]:
# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Create vector store
print("Creating vector store...")

vectorstore = Chroma.from_documents(
    documents,
    embeddings,
    collection_name="test_collection"
)

# Initialize retrievers
vector_search = VectorSearch(vectorstore)

bm25_search = BM25Search(documents)

hybrid_search = HybridRetriever(
    vectorstore,
    documents,
    vector_weight=0.7,
    bm25_weight=0.3
)

print("✓ All retrievers initialized")

Test Queries

In [ ]:
# Define test queries with expected relevant documents

test_queries = [
    {
        "query": "What is machine learning?",
        "relevant_ids": ["doc2", "doc4", "doc6"],
        "category": "AI"
    },
    {
        "query": "programming languages for web development",
        "relevant_ids": ["doc1", "doc3", "doc7"],
        "category": "Programming"
    },
    {
        "query": "database systems",
        "relevant_ids": ["doc5", "doc8"],
        "category": "Database"
    },
    {
        "query": "neural networks and deep learning",
        "relevant_ids": ["doc4", "doc2"],
        "category": "AI"
    },
    {
        "query": "JavaScript frameworks",
        "relevant_ids": ["doc7", "doc3"],
        "category": "Programming"
    }
]

print(f"Created {len(test_queries)} test queries")

Retrieval Performance Comparison

In [ ]:
# Compare retrieval methods

retrievers = {
    "Vector Search": vector_search,
    "BM25": bm25_search,
    "Hybrid": hybrid_search
}

results = []

metrics_calculator = RAGMetrics()

for query_data in test_queries:

    query = query_data["query"]
    relevant_ids = query_data["relevant_ids"]
    category = query_data["category"]

    print(f"\nQuery: '{query}'")
    print(f"Category: {category}")
    print(f"Expected relevant docs: {relevant_ids}")
    print("-" * 80)

    for method_name, retriever in retrievers.items():

        start_time = time.time()

        if method_name == "Hybrid":
            retrieved_docs = retriever.search(query)
        else:
            retrieved_docs = retriever.search(query, k=5)

        elapsed_time = time.time() - start_time

        retrieved_ids = [
            doc.metadata["id"]
            for doc in retrieved_docs
        ]

        metrics = metrics_calculator.calculate_retrieval_metrics(
            retrieved_ids,
            relevant_ids,
            k=5
        )

        results.append({
            "query": query,
            "category": category,
            "method": method_name,
            "precision": metrics["precision@k"],
            "recall": metrics["recall@k"],
            "f1": metrics["f1@k"],
            "mrr": metrics["mrr"],
            "map": metrics["map"],
            "ndcg": metrics["ndcg@k"],
            "time": elapsed_time,
            "retrieved": retrieved_ids[:3]
        })

        print(
            f"{method_name:.<20} "
            f"P@5: {metrics['precision@k']:.3f} | "
            f"R@5: {metrics['recall@k']:.3f} | "
            f"F1: {metrics['f1@k']:.3f} | "
            f"Time: {elapsed_time:.4f}s"
        )

        print(
            f"{'':.<20} Retrieved: {retrieved_ids[:3]}"
        )

df_results = pd.DataFrame(results)

print("\n✓ Evaluation complete")

Aggregate Performance Metrics

In [ ]:
avg_metrics = df_results.groupby(
    "method"
)[
    ["precision", "recall", "f1",
     "mrr", "map", "ndcg", "time"]
].mean()

print("\nAVERAGE PERFORMANCE METRICS")
print("=" * 80)

display(avg_metrics.round(4))

print("=" * 80)

In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(18, 10)
)

axes = axes.flatten()

metrics_to_plot = [
    "precision",
    "recall",
    "f1",
    "mrr",
    "map",
    "ndcg"
]

titles = [
    "Precision@5",
    "Recall@5",
    "F1@5",
    "MRR",
    "MAP",
    "NDCG@5"
]

for idx, (metric, title) in enumerate(
    zip(metrics_to_plot, titles)
):
    data = df_results.groupby("method")[metric].mean()

    data.plot(
        kind="bar",
        ax=axes[idx],
        color=["skyblue", "coral", "lightgreen"]
    )

    axes[idx].set_title(title)
    axes[idx].set_ylabel("Score")
    axes[idx].set_ylim(0, 1)

plt.tight_layout()
plt.show()

Performance by Query Category

In [ ]:
category_performance = (
    df_results
    .groupby(
        ["category", "method"]
    )[["precision", "recall", "f1"]]
    .mean()
)

print("\nPERFORMANCE BY QUERY CATEGORY")
print("=" * 80)

display(category_performance.round(3))

print("=" * 80)

Response Time Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

avg_time = (
    df_results.groupby("method")["time"]
    .mean()
)

avg_time.plot(
    kind="bar",
    ax=ax1,
    color=["skyblue", "coral", "lightgreen"]
)

ax1.set_title("Average Response Time")

df_results.boxplot(
    column="time",
    by="method",
    ax=ax2
)

plt.tight_layout()
plt.show()

display(
    df_results.groupby("method")["time"]
    .describe()
)

Top-K Analysis

In [ ]:
k_values = [1, 3, 5, 10]

test_query = test_queries[0]

topk_results = []

for k in k_values:

    for method_name, retriever in retrievers.items():

        if method_name == "Hybrid":
            retrieved_docs = retriever.search(
                test_query["query"]
            )
        else:
            retrieved_docs = retriever.search(
                test_query["query"],
                k=k
            )

        retrieved_ids = [
            doc.metadata["id"]
            for doc in retrieved_docs[:k]
        ]

        metrics = metrics_calculator.calculate_retrieval_metrics(
            retrieved_ids,
            test_query["relevant_ids"],
            k=k
        )

        topk_results.append({
            "k": k,
            "method": method_name,
            "precision": metrics["precision@k"],
            "recall": metrics["recall@k"],
            "f1": metrics["f1@k"]
        })

df_topk = pd.DataFrame(topk_results)

Hybrid Search Weight Analysis

In [ ]:
weight_combinations = [
    (1.0, 0.0),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8),
    (0.0, 1.0)
]

weight_results = []

for vector_weight, bm25_weight in weight_combinations:

    hybrid = HybridRetriever(
        vectorstore,
        documents,
        vector_weight=vector_weight,
        bm25_weight=bm25_weight
    )

    # Evaluation logic here

Summary and Recommendations

In [ ]:
print("=" * 80)
print(
    "RETRIEVAL STRATEGY ANALYSIS - SUMMARY"
    .center(80)
)
print("=" * 80)

ranking = avg_metrics.sort_values(
    "f1",
    ascending=False
)

print("\nPerformance Ranking:")
print(ranking[["precision", "recall", "f1"]])

print("\nSpeed Ranking:")
print(
    avg_metrics.sort_values("time")
)

print("=" * 80)

Export Results

In [ ]:
from datetime import datetime

export_dir = Path.cwd() / "exports"
export_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

df_results.to_csv(
    export_dir /
    f"retrieval_analysis_{timestamp}.csv",
    index=False
)

avg_metrics.to_csv(
    export_dir /
    f"retrieval_summary_{timestamp}.csv"
)

df_weights.to_csv(
    export_dir /
    f"hybrid_weights_{timestamp}.csv",
    index=False
)

print(
    f"✓ Results exported to {export_dir}"
)